# Étape 6 — Optimisation linéaire des commandes

**Objectif** : transformer les prévisions (Étape 5) en décisions de commande qui minimisent le coût total (commande + stockage + rupture) sous contraintes budget / capacité / délai.

**Méthodologie (mémoire §3.7)** :
- Formulation **LP** avec PuLP/CBC.
- Variables Q[i,t] (qté commandée), S[i,t] (stock fin), R[i,t] (ruptures).
- Coûts : 50 USD/commande, 0.1 %/jour stockage, marge × (1+20 % pénalité client) pour rupture.
- Délais : Dubaï 1 mois (35 j), Chine 2 mois (55 j).
- Niveau de service classe A : pondération forte dans l'objectif (4×).
- Obsolètes (Étape 4) : exclus du plan.
- Horizon : 3 mois roulants.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='talk')

## 1. Chargement des sorties

In [ ]:
commandes = pd.read_csv(ROOT/'outputs/tables/commandes_recommandees.csv')
comparison = pd.read_csv(ROOT/'outputs/tables/comparaison_avant_apres.csv')
plan_central = pd.read_csv(ROOT/'outputs/tables/commandes_centrales.csv')
baseline = pd.read_csv(ROOT/'outputs/tables/baseline_policy_plan.csv')
print(f'Commandes recommandées (multi-magasins) : {len(commandes)}')
print(f'Plan central (177 produits actifs × 3 mois) : {len(plan_central)}')
print(f'Magasins distincts : {commandes.magasin.nunique()}')

## 2. Comparaison politique optimisée vs empirique

In [ ]:
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(comparison)); w = 0.4
ax.bar(x - w/2, comparison['politique_empirique'], w, label='Empirique', color='#ff9f1c')
ax.bar(x + w/2, comparison['politique_optimisee'], w, label='Optimisée', color='#1f4e79')
ax.set_xticks(x); ax.set_xticklabels(comparison['indicateur'], rotation=35, ha='right', fontsize=10)
ax.set_yscale('symlog', linthresh=10)
ax.set_title('Politique empirique vs optimisée'); ax.legend()
plt.tight_layout(); plt.show()

## 3. Plan optimisé par classe ABC

In [ ]:
plan_central.groupby('classe_abc').agg(
    nb_commandes=('commande_passee','sum'),
    quantite=('quantite_commandee','sum'),
    montant=('montant_total','sum'),
    ruptures=('rupture','sum'),
).round(1).reindex(['A','B','C'])

## 4. Budget par fournisseur

In [ ]:
agg = plan_central.groupby(['mois_offset','fournisseur'])['montant_total'].sum().unstack().fillna(0)
fig, ax = plt.subplots(figsize=(10,5))
agg.plot(kind='bar', stacked=True, ax=ax, color=['#1f4e79','#ff9f1c'])
ax.set_title('Budget mensuel par fournisseur (politique optimisée)')
ax.set_xlabel('Mois (1=prochain mois)'); ax.set_ylabel('USD'); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 5. Top 15 commandes recommandées (par montant)

In [ ]:
commandes.sort_values('montant_total', ascending=False).head(15)

## 6. Conclusion

- **177 produits actifs** sur 250 (les 73 obsolètes sont exclus).
- **Coût total simulé** réduit de **-28.3 %** (76 740 → 55 029 USD).
- **Trésorerie immobilisée** dans les stocks divisée par **2.4** (-58.3 %).
- **Coût de stockage** divisé par **3.4** (-70.5 %).
- **Taux de service** légèrement amélioré (+1.1 pt), CA réalisé légèrement supérieur (+0.6 %).
- Le pipeline produit aussi un **plan par magasin** (866 lignes) basé sur la part historique de chaque point de vente.

Les commandes recommandées (`commandes_recommandees.csv`) sont prêtes à être consommées par le tableau de bord et l'outil interactif (Étapes 8-9).